# Focus Area 3 — Temperature Quality &amp; Microclimates
**Core Objective**: To demonstrate the advantages of high-resolution temperature data in capturing microclimates and computing derived metrics like PET, for better assessment of heat-related risks across Ethiopia, Tanzania, Eritrea, and Djibouti.

## Temperature data
- ERA5 Land data (primary — CBAM does not cover these countries)
- ERA5 data
- TAHMO data (sparse coverage — national met station CSV accepted where available)
<br>

EA: March 2024 heatwave

selected at Step 1 config cell (Ethiopia / Tanzania / Eritrea / Djibouti). Long-term trend covers 2015–2024.


Steps Breakdown — update to match what was actually built:

    Step 1: Setting up environment, authentication and country selection

    Step 2: Extract and visualise ERA5 temperature data (replaces TAHMO-first approach)

    Step 3: Extract ERA5 daily Tmin/Tmax and compare products

    Step 4: Load CBAM or pull ERA5 equivalent from GEE (CBAM fallback for new countries)

    Step 5: Compare ERA5 Tmin vs Tmax (CBAM vs ERA5 where CBAM available)

    Step 6: Compute PET and stress days with ERA5 (and CBAM where available)

    Step 7: Visualise heat change over 2015–2024 (was "last half a century" — shortened for compute time) **bold text**

In [2]:
# Configuration
Country = "Ethiopia"  # Options: 'Ethiopia', 'Tanzania', 'Eritrea', 'Djibouti'

Country_region = {
    'Ethiopia': [33.0, 3.0, 48.0, 15.0],
    'Tanzania': [29.0, -12.0, 41.0, -1.0],
    'Eritrea': [36.5, 12.5, 43.5, 18.0],
    'Djibouti': [41.5, 10.9, 43.5, 12.7]
}

##**Step 1: Setting up environment and Authentication**

In [ ]:
# @title 1a) Setting up environment installing required Dependencies {"display-mode":"form"}
# @markdown This cell installs the required python dependencies and functions for the notebook.<br>
# @markdown If you encounter any errors, please restart the runtime and try again. <br>

print("Installing required dependencies...")
!pip install git+https://github.com/TAHMO/NOAA.git > /dev/null 2>&1

!jupyter nbextension enable --py widgetsnbextension > /dev/null 2>&1

print("✅ Dependencies installed successfully.")

print("Importing required libraries...")
import pandas as pd
import matplotlib.pyplot as plt
import os
import ee
import xarray as xr
import numpy as np
from scipy.stats import pearsonr, ttest_rel
import random
import json
import seaborn as sns

from utils.ground_stations import plot_stations_folium
from utils.helpers import get_region_geojson
from utils.CHIRPS_helpers import get_chirps_pentad_gee
from utils.CBAM_helpers import CBAMClient, extract_cbam_data # CBAM helper functions
from utils.plotting import select, scale, plot_xarray_data, plot_xarray_data2, compare_xarray_datasets, compare_xarray_datasets2 # Plotting helper functions
from utils.IMERG_helpers import get_imerg_raw
from utils.ERA5_helpers import era5_data_extracts, era5_var_handling
from utils.filter_stations import RetrieveData

import cartopy.crs as ccrs
import cartopy.feature as cfeature

%matplotlib inline

print("✅ Libraries imported successfully.")

def build_xr_from_stations(ds, stations_metadata, var_name=None):
    # Auto-detect variable if not provided
    if var_name is None:
        candidate_vars = ['total_precipitation', 'total_rainfall', 'precipitation']
        found = [v for v in candidate_vars if v in ds.data_vars]
        if not found:
            raise ValueError(f"None of expected precipitation variable names {candidate_vars} found in dataset vars: {list(ds.data_vars)}")
        var_name = found[0]

    # Determine dimension names
    if {'x', 'y'}.issubset(ds.dims):
        lon_dim, lat_dim = 'x', 'y'
    elif {'lon', 'lat'}.issubset(ds.dims):
        lon_dim, lat_dim = 'lon', 'lat'
    else:
        raise ValueError(f"Dataset dims {list(ds.dims)} do not contain expected (x,y) or (lon,lat).")

    all_stations_data = {}
    for _, row in stations_metadata.iterrows():
        station_code = row['code']
        lat = float(row['lat'])
        lon = float(row['lon'])
        # Skip stations outside domain (quick bounds check)
        if not (ds[lon_dim].min() <= lon <= ds[lon_dim].max() and ds[lat_dim].min() <= lat <= ds[lat_dim].max()):
            continue
        station_da = ds[var_name].sel({lon_dim: lon, lat_dim: lat}, method="nearest")
        station_df = station_da.to_dataframe(name=station_code)
        all_stations_data[station_code] = station_df[station_code]

    combined_df = pd.DataFrame(all_stations_data)
    return combined_df

def plot_temperatures(tmin_df, tavg_df, tmax_df, station_code=None):
    """
    Plots the daily minimum, average, and maximum temperatures for a specified TAHMO station.
    """
    if station_code is None:
        station_code = random.choice(tmin_df.columns.tolist())
        print(f"Randomly selected station: {station_code}")
    elif station_code not in tmin_df.columns:
        print(f"Station code {station_code} not found in the data.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(tmin_df.index, tmin_df[station_code], label='Min Temp', linestyle='-')
    plt.plot(tavg_df.index, tavg_df[station_code], label='Avg Temp', linestyle='-')
    plt.plot(tmax_df.index, tmax_df[station_code], label='Max Temp', linestyle='-')

    plt.xlabel('Date')
    plt.ylabel('Temperature (°C)')
    plt.title(f'Daily Temperatures for Station {station_code}')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title ### 1b) Configuration & Earth Engine Authentication {"display-mode":"form"}
# @markdown This step loads your TAHMO API credentials and initializes Google Earth Engine. <br>
# @markdown **1. Config File:** Upload the `config.json` file provided in the workshop materials. <br>
# @markdown **2. Earth Engine:** You must provide your own Google Cloud Project ID below. <br>
# @markdown If you do not have one, please create a free project here: [Create GCP Project](https://console.cloud.google.com/projectcreate) <br>
# @markdown Then, enable the Earth Engine API for your project: [Enable GEE API](https://console.cloud.google.com/apis/api/earthengine.googleapis.com/overview) <br>

import os
import json
import ee

# ── 1. Load Config File ──────────────────────────────────────────────────────
from google.colab import files

print("📂 Please upload your config.json file:")
uploaded = files.upload()

config = None
for filename in uploaded.keys():
    if filename.endswith('.json'):
        with open(filename, 'r') as f:
            config = json.load(f)
        print(f"✅ Config file loaded successfully from '{filename}'")

        # Rename to config.json for consistency in later cells
        if filename != 'config.json':
            os.rename(filename, 'config.json')
        break

if config is None:
    print("❌ No JSON file uploaded. Please re-run the cell and upload a .json file.")

# ── 2. Authenticate Google Earth Engine ──────────────────────────────────────
print("\n🔐 Authenticating Google Earth Engine...")
try:
    ee.Authenticate()
    print("✅ GEE Authentication successful.")
except Exception as e:
    print(f"❌ Authentication failed: {e}")

# ── 3. Prompt for Project ID ─────────────────────────────────────────────────
print("\n" + "="*60)
print("⚙️  GOOGLE CLOUD PROJECT SETUP")
print("="*60)
print("Please enter your Google Cloud Project ID.")
print("(You can find this in the GCP Console dashboard)")
print("-" * 60)

# Prompt the user for input
user_project_id = input("Enter your GCP Project ID (e.g., 'my-gee-project-123'): ").strip()

if not user_project_id:
    print("❌ No Project ID provided. Please re-run the cell and enter a valid Project ID.")
    GEE_PROJECT = "your-project-id-here"
else:
    # Set both variables to ensure compatibility across all Focus Areas
    GEE_PROJECT = user_project_id
    project_id = GEE_PROJECT  # Alias for cells that specifically look for 'project_id'
    print(f"✅ Using Project ID: {GEE_PROJECT}")

# ── 4. Initialize Earth Engine ───────────────────────────────────────────────
print(f"\n🚀 Initializing Earth Engine with project: {GEE_PROJECT}...")
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Google Earth Engine initialized successfully!")
except ee.EEException as e:
    if "PERMISSION_DENIED" in str(e):
        print(f"❌ Earth Engine initialization failed due to PERMISSION_DENIED.")
        print(f"Please ensure the Earth Engine API is enabled for your project:")
        print(f"https://console.cloud.google.com/apis/api/earthengine.googleapis.com/overview?project={GEE_PROJECT}")
    elif "NONCOMMERCE" in str(e) or "not registered" in str(e).lower():
        print(f"❌ Earth Engine initialization failed.")
        print("Please ensure you have registered your project for Earth Engine:")
        print("https://console.cloud.google.com/earth-engine/configuration")
    else:
        print(f"❌ Earth Engine initialization failed: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during initialization: {e}")

## **Step 2: Extract and visualise TAHMO temperature data**

In [ ]:
# @title 2a) Visualise your selected region {"display-mode":"form"}
# @markdown This cell previews the bounding box set at the first configuration section
import time
import json
import plotly.graph_objects as go
import geopandas as gpd
from shapely.geometry import Polygon
import sys
import importlib
import ipywidgets as widgets
from IPython.display import display

# --- Environment Detection ---
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()
print(f"💡 Running in {'Google Colab' if IS_COLAB else 'Local Jupyter'} environment.")

# Load config from the current working directory (uploaded in Step 1b)
try:
    with open('config.json', 'r') as f:
        config = json.load(f)
    location_key = config.get('location_keys', None)
except Exception:
    location_key = None

def xmin_ymin_xmax_ymax(polygon):
    lons = [pt[0] for pt in polygon]
    lats = [pt[1] for pt in polygon]
    return min(lons), min(lats), max(lons), max(lats)

def fetch_region_google(query):
    """Primary: Fetch polygon geometry via Google Maps API"""
    if not location_key:
        raise RuntimeError("Missing Google Maps API key.")
    region_geom = get_region_geojson(query, location_key)['geometry']['coordinates'][0]
    return region_geom

def fetch_region_osm(query):
    """Fallback: Fetch geometry from OSM (Nominatim) via GeoPandas"""
    url = f"https://nominatim.openstreetmap.org/search?country={query}&format=geojson&polygon_geojson=1"
    gdf = gpd.read_file(url)
    if gdf.empty:
        raise ValueError("No OSM data found for that query.")
    geom = gdf.iloc[0].geometry
    if geom.geom_type == "Polygon":
        return list(geom.exterior.coords)
    elif geom.geom_type == "MultiPolygon":
        polygons = list(geom.geoms)
        if polygons:
            largest_polygon = max(polygons, key=lambda p: len(p.exterior.coords))
            return list(largest_polygon.exterior.coords)
        else:
             raise ValueError("No polygons found in MultiPolygon from OSM.")
    else:
        raise ValueError("Unsupported geometry type from OSM.")

def show_region_plotly(polygon, region_name="Region", margin=0.05):
    """Plot polygon with Plotly Mapbox"""
    lons = [pt[0] for pt in polygon]
    lats = [pt[1] for pt in polygon]
    fig = go.Figure(go.Scattermapbox(
        lon=lons + [lons[0]],
        lat=lats + [lats[0]],
        mode="lines",
        fill="toself",
        fillcolor="rgba(0,0,255,0.3)",
        name=region_name
    ))
    fig.update_layout(
        mapbox_style="open-street-map",
        mapbox=dict(center={"lat": sum(lats)/len(lats), "lon": sum(lons)/len(lons)}, zoom=5),
        margin=dict(r=40, t=30, l=0, b=0),
        title=f"Region of Interest: {region_name}",
        height=300,
        width=500
    )
    fig.show()
    return fig

# --- Determine Region Geometry ---
if Country_region.get(Country) is not None:
    region_geom = Country_region[Country]

    # FIX: Convert flat bounding box [xmin, ymin, xmax, ymax] to polygon list of tuples
    if len(region_geom) == 4 and all(isinstance(p, (int, float)) for p in region_geom):
        xmin, ymin, xmax, ymax = region_geom
        region_geom = [(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax), (xmin, ymin)]
else:
    region_geom = fetch_region_osm(Country)

if region_geom:
    xmin, ymin, xmax, ymax = xmin_ymin_xmax_ymax(region_geom)
    show_region_plotly(region_geom, region_name=Country)
    print(f"📦 Bounding box -> xmin: {xmin}, ymin: {ymin}, xmax: {xmax}, ymax: {ymax}")
else:
    print("🛑 No geometry available.")

In [ ]:
# @title 2b) Extract TAHMO Metadata {"display-mode":"form"}
# @markdown This cell extracts TAHMO station metadata for the selected country/region.

from utils.filter_stations import RetrieveData
import os
import time
import plotly.express as px
import pandas as pd
from shapely.geometry import Point, Polygon

region_query = Country
dir_path = './data'
os.makedirs(dir_path, exist_ok=True)

print(f"✅ Data directory created at: {dir_path}")

def plot_stations_plotly(dataframes, colors=None, zoom=5, height=400,
                         width=700, legend_title='Station Locations'):
    """
    Plot stations from one or more dataframes on a Plotly mapbox.
    Dynamically handles both TAHMO ('location.latitude') and custom ('lat') column names.
    """
    if colors is None:
        colors = ["blue", "red", "green", "purple", "orange"]

    frames = []
    for i, df in enumerate(dataframes):
        if df.empty:
            continue
        temp = df.copy()
        temp["color"] = colors[i % len(colors)]

        # Normalize column names for plotting
        if 'location.latitude' in temp.columns:
            temp.rename(columns={'location.latitude': 'lat', 'location.longitude': 'lon', 'code': 'station_id'}, inplace=True)

        frames.append(temp)

    if not frames:
        print("⚠️ No stations to plot.")
        return None

    combined = pd.concat(frames, ignore_index=True)

    # Ensure 'station_id' exists for hover
    if 'station_id' not in combined.columns:
        if 'code' in combined.columns:
            combined.rename(columns={'code': 'station_id'}, inplace=True)
        else:
            combined['station_id'] = 'Station'

    fig = px.scatter_mapbox(
        combined,
        lat='lat',
        lon='lon',
        color="color",
        hover_name="station_id",
        zoom=zoom,
        height=height,
        width=width
    )

    fig.update_layout(
        mapbox_style="open-street-map",
        legend_title=legend_title,
        margin={"r": 0, "t": 30, "l": 0, "b": 0}
    )

    return fig

api_key = config.get('apiKey')
api_secret = config.get('apiSecret')

if not api_key or not api_secret:
    print("❌ API credentials not found in config. Please check your config.json.")
else:
    # Initialize the class
    rd = RetrieveData(apiKey=api_key, apiSecret=api_secret)

    # Extracting TAHMO data
    print("Extracting TAHMO data...")
    info = rd.get_stations_info()

    # Use region_geom as the polygon for filtering
    if 'region_geom' in locals() and region_geom:
        # Create a Shapely Polygon from the region_geom coordinates
        region_polygon = Polygon(region_geom)

        # Filter stations that are within the polygon
        info = info[info.apply(lambda row: region_polygon.contains(Point(row['location.longitude'], row['location.latitude'])), axis=1)]

        if info.empty:
            print("⚠️ No TAHMO stations found within the selected region bounding box.")
        else:
            print(f"✅ Found {len(info)} TAHMO stations within the region.")
    else:
        print("🛑 No region geometry available to filter stations.")
        info = pd.DataFrame() # Ensure info is an empty dataframe if no geometry

    print(f"Total number of stations found: {len(info)}")

    if not info.empty:
        # save the data as csv to the created directory
        csv_path = os.path.join(dir_path, f'tahmo_metadata_{region_query}.csv')
        info.to_csv(csv_path, index=False)
        print(f"💾 Saved metadata to {csv_path}")

        # wait for 2 seconds before visual
        time.sleep(2)

        # Visualise the data
        fig = plot_stations_plotly([info])
        if fig:
            fig.show()
    else:
        print("ℹ️ Skipping CSV save and visualization since no stations were found.")

In [ ]:
# @title 2c) Extract the TAHMO temperature 5 minute data for 2024 and get the tmin, tavg and tmax {"display-mode":"form"}
# Load TAHMO EAC stations previously extracted
# eac_metadata = pd.read_csv(f'{dir_path}/tahmo_metadata_{region_query}.csv')
# eac_metadata = eac_metadata[['code',
#                              'location.latitude',
#                              'location.longitude']].rename(columns={'location.latitude': 'lat',
#                                                                     'location.longitude': 'lon'})

# # Initialize the class
# rd = RetrieveData(apiKey=api_key,
#                   apiSecret=api_secret)

# print('Extracting Temperature data ...')
# # # Get the temperature data for the EAC stations in 5min intervals
# # eac_temp = rd.multiple_measurements(stations_list=eac_metadata['code'].tolist(),
# #                                      startDate=start_date,
# #                                      endDate=end_date,
# #                                      variables=['te'],
# #                                      csv_file = f'{dir_path}/tahmo_temp_{region_query}.csv',
# #                                      aggregate='5min'
# #                                      )


# # # Aggregate the values to get the min, mean and max for the day
# # tahmo_eac_tmin = rd.aggregate_variables(
# #     eac_temp,
# #     freq='1D',
# #     method='min'
# # )
# # tahmo_eac_tavg = rd.aggregate_variables(
# #     eac_temp,
# #     freq='1D',
# #     method='mean'
# # )
# # tahmo_eac_tmax = rd.aggregate_variables(
# #     eac_temp,
# #     freq='1D',
# #     method='max'
# # )


# # # plot_temperatures(tahmo_eac_tmin, tahmo_eac_tavg, tahmo_eac_tmax)


# # # Save the variables
# # tahmo_eac_tmin.to_csv(f'{dir_path}/tahmo_tmin_{region_query}.csv', index=True)
# # tahmo_eac_tavg.to_csv(f'{dir_path}/tahmo_tmin_{region_query}.csv', index=True)
# # tahmo_eac_tmax.to_csv(f'{dir_path}/tahmo_tmin_{region_query}.csv', index=True)

# # Method to cleanup the data
# def format_cleanup(df, localize_none=True):
#   # Rename Unnamed: 0 to Date
#   df.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
#   # Set Date as index
#   df.set_index('Date', inplace=True)

#   # convert Date to datetime
#   df.index = pd.to_datetime(df.index)
#   if localize_none:
#     # set tz_localize to None
#     df.index = df.index.tz_localize(None)
#   return df

# # get only stations that have the metadata
# def match_with_metadata(df, metadata, column='code', localize_none=True):
#   # Format the data
#   df = format_cleanup(df, localize_none=localize_none)

#   # get the stations list
#   stations_list = metadata[column].to_list()

#   # Subset the columns from the dataframe with the data
#   df = df[df.columns.intersection(stations_list)]

#   return df

# # Load the tahmo data
# base_data_path = '/content/drive/Shareddrives/NOAA-workshop2/Temperature/ground'
# tahmo_tmin = pd.read_csv(os.path.join(base_data_path,'eac_tmin_march_2024.csv' ))
# tahmo_tmin = match_with_metadata(tahmo_tmin, info)
# tahmo_tmax = pd.read_csv(os.path.join(base_data_path,'eac_tmax_march_2024.csv' ))
# tahmo_tmax = match_with_metadata(tahmo_tmax, info)
# tahmo_tavg = pd.read_csv(os.path.join(base_data_path,'eac_tavg_march_2024.csv' ))
# tahmo_tavg = match_with_metadata(tahmo_tavg, info)

# # check if the data is well loaded
# start_date = tahmo_tavg.index.min().strftime('%Y-%m-%d')
# end_date = tahmo_tavg.index.max().strftime('%Y-%m-%d')

# print('✅ Tahmo data loaded')
# # visualise the tavg data
# print('Printing the first 5 rows of the tavg data')
# tahmo_tavg.head().dropna(axis=1)
# @title 2c) Load/Extract the TAHMO temperature data for March 2024 and get the tmin, tavg and tmax {"display-mode":"form"}
# @markdown This cell loads the TAHMO temperature data for March 2024.
# @markdown If the pre-extracted files are not found, it attempts to extract them from the TAHMO API.
import os
import pandas as pd

# Ensure the metadata from Step 2b is available
if 'info' not in locals() or info.empty:
    print("⚠️ Metadata 'info' not found. Loading from CSV...")
    # Use dir_path from Step 2b (which is './data')
    info = pd.read_csv(f'{dir_path}/tahmo_metadata_{region_query}.csv')
    info = info[['code', 'location.latitude', 'location.longitude']].rename(
        columns={'location.latitude': 'lat', 'location.longitude': 'lon'}
    )

# Define paths for the data and FORCE create the intended directory
base_data_path = './data/temperature'
os.makedirs(base_data_path, exist_ok=True)

# Use region_query for file naming to avoid conflicts between different countries
tmin_path = os.path.join(base_data_path, f'tahmo_tmin_march_2024_{region_query}.csv')
tmax_path = os.path.join(base_data_path, f'tahmo_tmax_march_2024_{region_query}.csv')
tavg_path = os.path.join(base_data_path, f'tahmo_tavg_march_2024_{region_query}.csv')

# Helper functions
def format_cleanup(df, localize_none=True):
    """Clean up the dataframe: rename index, set as datetime, remove tz."""
    if 'Unnamed: 0' in df.columns:
        df.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
    if 'Date' in df.columns:
        df.set_index('Date', inplace=True)
    df.index = pd.to_datetime(df.index)
    if localize_none:
        df.index = df.index.tz_localize(None)
    return df

def match_with_metadata(df, metadata, column='code', localize_none=True):
    """Format data and subset columns to match available metadata."""
    df = format_cleanup(df, localize_none=localize_none)
    stations_list = metadata[column].to_list()
    # Keep only columns that are in the metadata
    df = df[df.columns.intersection(stations_list)]
    return df

# Check if files exist, if not, try to extract them
if not (os.path.exists(tmin_path) and os.path.exists(tmax_path) and os.path.exists(tavg_path)):
    print("ℹ️ Pre-extracted files not found. Attempting to extract from TAHMO API...")
    try:
        # Initialize the class
        rd = RetrieveData(apiKey=api_key, apiSecret=api_secret)
        print('Extracting Temperature data for March 2024 ...')

        # Define date range for March 2024
        start_date = '2024-03-01'
        end_date = '2024-03-31'

        eac_temp = rd.multiple_measurements(
            stations_list=info['code'].tolist(),
            startDate=start_date,
            endDate=end_date,
            variables=['te'],
            csv_file=os.path.join(base_data_path, f'tahmo_temp_march_2024_{region_query}.csv'),
            aggregate='5min'
        )

        # Aggregate to daily min, mean, max
        tahmo_tmin = rd.aggregate_variables(eac_temp, freq='1D', method='min')
        tahmo_tavg = rd.aggregate_variables(eac_temp, freq='1D', method='mean')
        tahmo_tmax = rd.aggregate_variables(eac_temp, freq='1D', method='max')

        # Save the extracted data
        tahmo_tmin.to_csv(tmin_path, index=True)
        tahmo_tavg.to_csv(tavg_path, index=True)
        tahmo_tmax.to_csv(tmax_path, index=True)
        print(f"✅ Extracted and saved TAHMO data to: {base_data_path}")

    except Exception as e:
        print(f"❌ Could not extract data from API: {e}")
        print("Please ensure the pre-extracted CSV files are available.")
else:
    print(f"✅ Loading pre-extracted TAHMO data from: {base_data_path}")

# Load the data
tahmo_tmin = pd.read_csv(tmin_path)
tahmo_tmin = match_with_metadata(tahmo_tmin, info)

tahmo_tmax = pd.read_csv(tmax_path)
tahmo_tmax = match_with_metadata(tahmo_tmax, info)

tahmo_tavg = pd.read_csv(tavg_path)
tahmo_tavg = match_with_metadata(tahmo_tavg, info)

# Check if the data is well loaded
if not tahmo_tavg.empty:
    start_date = tahmo_tavg.index.min().strftime('%Y-%m-%d')
    end_date = tahmo_tavg.index.max().strftime('%Y-%m-%d')
    print(f'✅ Tahmo data loaded successfully ({start_date} to {end_date})')

    # Visualise the tavg data
    print('\nPrinting the first 5 rows of the tavg data:')
    display(tahmo_tavg.head().dropna(axis=1))
else:
    print("⚠️ TAHMO temperature data is empty or could not be loaded.")


In [ ]:
# @title 2d) Randomly visualise TAHMO Tmin, Tavg, and Tmax data on a single chart {"display-mode":"form"}
# @markdown Rerun this cell to skip to the next ground station
# @markdown This cell calls the existing `plot_temperatures` function to visualize the data.

# Check if the data variables exist and are not empty before plotting
if 'tahmo_tmin' in locals() and 'tahmo_tavg' in locals() and 'tahmo_tmax' in locals():
    if not tahmo_tmin.empty and not tahmo_tavg.empty and not tahmo_tmax.empty:
        # Call the existing plot_temperatures function
        plot_temperatures(tahmo_tmin, tahmo_tavg, tahmo_tmax)
    else:
        print("⚠️ One or more temperature DataFrames (tmin, tavg, tmax) are empty. Please check Step 2c.")
else:
    print("⚠️ Temperature variables (tahmo_tmin, tahmo_tavg, tahmo_tmax) not found. Please run Step 2c first.")

##**Step 3: Extract ERA5 data and compare with ground data**

In [ ]:
# @title Step 5: ERA5 daily temperature — load or pull from GEE
import os, glob
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import pandas as pd, numpy as np, xarray as xr
from IPython.display import HTML, display

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — adjust these if your notebook sets them elsewhere
# ─────────────────────────────────────────────────────────────────────────────
era5_base_path = './data/era5'
os.makedirs(era5_base_path, exist_ok=True)

TEMP_START = '2024-03-01'
TEMP_END   = '2024-03-31'
# GEE_PROJECT = "natural-notch-435413-j3"   # same as rainfall notebook

# Spatial bounds — use whatever xmin/xmax/ymin/ymax are set upstream
# Falls back to a broad East Africa bbox if not defined
try:
    _xmin, _xmax, _ymin, _ymax = xmin, xmax, ymin, ymax
except NameError:
    _xmin, _xmax, _ymin, _ymax = 28.0, 48.0, -5.0, 16.0
    print(f'ℹ️  xmin/xmax/ymin/ymax not set — using East Africa default: '
          f'[{_xmin},{_xmax},{_ymin},{_ymax}]')

# ─────────────────────────────────────────────────────────────────────────────
# GEE PULLER: ERA5 Land daily temperature → NetCDF
# ERA5 Land temperatures are in Kelvin — converted to °C here
# Variables: temperature_2m (mean), temperature_2m_min, temperature_2m_max
# ─────────────────────────────────────────────────────────────────────────────
def pull_era5_temp(out_nc, gee_band, label, start, end, bbox):
    """
    Pull a single ERA5 Land temperature variable from GEE and save as NetCDF.
    Converts Kelvin → Celsius.  One image per day.
    ⏱  ~5-10 min for one month at 0.1° resolution.
    """
    import ee
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)

    try:
        import geemap
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'geemap', '-q'], check=True)
        import geemap

    from tqdm.notebook import tqdm

    lon_min, lat_min, lon_max, lat_max = bbox
    roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

    col = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
          .filterDate(start, end)
          .filterBounds(roi)
          .select(gee_band)
          .map(lambda img: img
               .subtract(273.15)            # K → °C
               .rename(gee_band)
               .copyProperties(img, ['system:time_start']))
    )
    n = col.size().getInfo()
    print(f'   {label}: {n} daily images found ({start} → {end})')

    tmp_dir = f'./tmp/era5_{label}'
    os.makedirs(tmp_dir, exist_ok=True)
    img_list = col.toList(n)

    for i in tqdm(range(n), desc=f'ERA5 {label}', unit='day'):
        img      = ee.Image(img_list.get(i))
        date_str = img.date().format('yyyyMMdd').getInfo()
        out_file = os.path.join(tmp_dir, f'{date_str}.tif')
        if os.path.exists(out_file):
            continue
        geemap.ee_export_image(
            img, filename=out_file,
            scale=11132, region=roi, file_per_band=False
        )

    # ── Assemble TIFFs → NetCDF (rasterio-free method) ───────────────────────
    import rasterio as rio
    tiffs = sorted(glob.glob(os.path.join(tmp_dir, '*.tif')))
    print(f'   Assembling {len(tiffs)} TIFFs...')

    with rio.open(tiffs[0]) as src:
        transform = src.transform
        height, width = src.height, src.width

    lons = np.array([transform.c + (i + 0.5) * transform.a for i in range(width)])
    lats = np.array([transform.f + (j + 0.5) * transform.e for j in range(height)])

    stack = np.zeros((len(tiffs), height, width), dtype=np.float32)
    for i, tif in enumerate(tiffs):
        with rio.open(tif) as src:
            stack[i] = src.read(1).astype(np.float32)

    dates = pd.to_datetime(
        [os.path.basename(f).split('.')[0] for f in tiffs], format='%Y%m%d'
    )
    ds = xr.Dataset(
        {gee_band: (['time', 'lat', 'lon'], stack)},
        coords={'time': dates, 'lat': lats, 'lon': lons}
    )
    ds = ds.where(ds > -200)   # mask fill values (K→°C fill ≈ -9726)
    ds.to_netcdf(out_nc)
    print(f'   ✅ Saved: {out_nc}')
    return ds


# ─────────────────────────────────────────────────────────────────────────────
# LOAD OR PULL: tavg, tmin, tmax
# ─────────────────────────────────────────────────────────────────────────────
ERA5_FILES = {
    'tavg': {
        'nc':   os.path.join(era5_base_path, 'era5_tavg_march_2024.nc'),
        'band': 'temperature_2m',
        'label': 'tavg',
    },
    'tmin': {
        'nc':   os.path.join(era5_base_path, 'era5_tmin_march_2024.nc'),
        'band': 'temperature_2m_min',
        'label': 'tmin',
    },
    'tmax': {
        'nc':   os.path.join(era5_base_path, 'era5_tmax_march_2024.nc'),
        'band': 'temperature_2m_max',
        'label': 'tmax',
    },
}

bbox = [_xmin, _ymin, _xmax, _ymax]

era5_loaded = {}
for key, cfg in ERA5_FILES.items():
    if os.path.exists(cfg['nc']):
        print(f'✅ {key}: loading from local cache')
        era5_loaded[key] = xr.open_dataset(cfg['nc'])
    else:
        print(f'⚠️  {key}: not found — pulling from GEE...')
        era5_loaded[key] = pull_era5_temp(
            out_nc=cfg['nc'],
            gee_band=cfg['band'],
            label=cfg['label'],
            start=TEMP_START,
            end=TEMP_END,
            bbox=bbox
        )

era5_tavg = era5_loaded['tavg']
era5_tmin = era5_loaded['tmin']
era5_tmax = era5_loaded['tmax']

# ─────────────────────────────────────────────────────────────────────────────
# SUBSET HELPER (unchanged from original — handles ascending/descending lat)
# ─────────────────────────────────────────────────────────────────────────────
def subset_within_x_ymin_max(ds, xmin, xmax, ymin, ymax):
    if ds.lat.values[0] > ds.lat.values[-1]:
        lat_slice = slice(ymax, ymin)
    else:
        lat_slice = slice(ymin, ymax)
    return ds.sel(lat=lat_slice, lon=slice(xmin, xmax))

era5_tavg = subset_within_x_ymin_max(
    era5_tavg, _xmin, _xmax, _ymin, _ymax
).sel(time=slice('2024-03-01', '2024-03-05'))

era5_tmin = subset_within_x_ymin_max(
    era5_tmin, _xmin, _xmax, _ymin, _ymax
).sel(time=slice('2024-03-01', '2024-03-05'))

era5_tmax = subset_within_x_ymin_max(
    era5_tmax, _xmin, _xmax, _ymin, _ymax
).sel(time=slice('2024-03-01', '2024-03-05'))

# Auto-detect temperature variable name
era5_var_name = next(
    (v for v in era5_tmax.data_vars
     if 'temp' in v.lower() or 't2m' in v.lower()),
    list(era5_tmax.data_vars)[0]
)
print(f'\n✅ ERA5 temperature ready. Variable: "{era5_var_name}"')
for key, ds in [('tavg', era5_tavg), ('tmin', era5_tmin), ('tmax', era5_tmax)]:
    print(f'   {key}: {len(ds.time)} days | '
          f'{float(ds[list(ds.data_vars)[0]].min()):.1f}°C – '
          f'{float(ds[list(ds.data_vars)[0]].max()):.1f}°C')

# ─────────────────────────────────────────────────────────────────────────────
# GUARD: check tahmo_tmax and info exist before animation
# ─────────────────────────────────────────────────────────────────────────────
_tahmo_ok = 'tahmo_tmax' in dir() and tahmo_tmax is not None and not tahmo_tmax.empty
_info_ok  = 'info' in dir() and info is not None and not info.empty

if not _tahmo_ok:
    print('\n⚠️  tahmo_tmax not found — run the TAHMO temperature loading cell first.')
    print('   Skipping animation. ERA5 data is ready for when it is available.')
elif not _info_ok:
    print('\n⚠️  Station metadata (info) not found — run the metadata loading cell first.')
else:
    # ── Animation (unchanged from original) ──────────────────────────────────
    def point_plot(
        weather_df, metadata_df, variable_name='Observation',
        cmap='viridis', robust=True, fig_title=None, interval=300,
        bbox=None, save=False, metadata_columns=None,
        grid_da=None, grid_cmap='coolwarm', grid_alpha=0.6, grid_da_var=None
    ):
        if metadata_columns is None:
            metadata_columns = ['station_id', 'lat', 'lon']
        station_col, lat_col, lon_col = metadata_columns

        for col in [station_col, lat_col, lon_col]:
            if col not in metadata_df.columns:
                raise ValueError(f'Missing metadata column: {col}')

        if weather_df.columns.name != station_col:
            weather_df.columns.name = station_col

        lons = metadata_df.set_index(station_col).loc[weather_df.columns][lon_col].values
        lats = metadata_df.set_index(station_col).loc[weather_df.columns][lat_col].values

        data_values = weather_df.values.flatten()
        vmin = np.nanpercentile(data_values, 2 if robust else 0)
        vmax = np.nanpercentile(data_values, 98 if robust else 100)
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

        fig = plt.figure(figsize=(8, 5))
        ax  = plt.axes(projection=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')

        if bbox is not None:
            ax.set_extent(bbox)
        else:
            pad = 0.5
            ax.set_extent([lons.min()-pad, lons.max()+pad,
                           lats.min()-pad, lats.max()+pad],
                          crs=ccrs.PlateCarree())

        grid_plot = None
        if grid_da is not None:
            if isinstance(grid_da, xr.Dataset):
                if grid_da_var is None or grid_da_var not in grid_da.data_vars:
                    grid_da_var = list(grid_da.data_vars)[0]
                grid_data_to_plot = grid_da[grid_da_var]
            elif isinstance(grid_da, xr.DataArray):
                grid_data_to_plot = grid_da
                grid_da_var = grid_da.name or 'Grid'
            else:
                raise TypeError('grid_da must be xr.Dataset or xr.DataArray')

            init_frame = (grid_data_to_plot.isel(time=0)
                          if 'time' in grid_data_to_plot.dims
                          else grid_data_to_plot)
            grid_plot = init_frame.plot(
                ax=ax, transform=ccrs.PlateCarree(),
                cmap=grid_cmap, alpha=grid_alpha,
                add_colorbar=True,
                cbar_kwargs={'shrink': 0.8, 'pad': 0.02, 'label': grid_da_var}
            )

        scatter = ax.scatter(
            lons, lats,
            c=weather_df.iloc[0].values,
            cmap=cmap, norm=norm, s=50,
            transform=ccrs.PlateCarree(),
            edgecolor='black', linewidth=0.3, zorder=3
        )
        plt.tight_layout()
        cbar = plt.colorbar(scatter, ax=ax, orientation='vertical',
                            shrink=0.8, pad=0.02)
        cbar.set_label(variable_name, fontsize=10)

        time_index = weather_df.index
        title = ax.set_title(
            f'{fig_title or variable_name}\n'
            f'{time_index[0].strftime("%Y-%m-%d") if hasattr(time_index[0], "strftime") else time_index[0]}',
            fontsize=12
        )

        def update(frame):
            scatter.set_array(weather_df.iloc[frame].values)
            if grid_plot is not None and 'time' in grid_data_to_plot.dims:
                if ax.images:
                    ax.images[-1].remove()
                grid_data_to_plot.isel(time=frame).plot(
                    ax=ax, transform=ccrs.PlateCarree(),
                    cmap=grid_cmap, alpha=grid_alpha,
                    add_colorbar=False, zorder=1
                )
            label = (time_index[frame].strftime('%Y-%m-%d')
                     if hasattr(time_index[frame], 'strftime')
                     else str(time_index[frame]))
            title.set_text(f'{fig_title or variable_name}\n{label}')
            return [scatter, title] + ax.images

        ani = animation.FuncAnimation(
            fig, update, frames=len(weather_df),
            interval=interval, blit=False
        )
        plt.close(fig)
        if save:
            ani.save(f'{(fig_title or variable_name).replace(" ","_")}.gif',
                     writer='pillow', fps=3, dpi=150)
        return HTML(ani.to_jshtml())

    # ── Run animation ─────────────────────────────────────────────────────────
    common_times = tahmo_tmax.index.intersection(era5_tmax.time.values)
    if len(common_times) == 0:
        print('⚠️  No common times between TAHMO and ERA5. '
              'Check that both cover 2024-03-01 → 2024-03-05.')
    else:
        html_anim = point_plot(
            tahmo_tmax.loc[common_times],
            info,
            variable_name='Ground Tmax (°C)',
            metadata_columns=['code', 'location.latitude', 'location.longitude'],
            cmap='plasma',
            grid_da=era5_tmax.sel(time=common_times),
            grid_cmap='coolwarm',
            grid_alpha=0.5,
            fig_title='Station Tmax vs ERA5 Background',
            grid_da_var=era5_var_name,
        )
        print('✅ Animation generated')
        display(html_anim)

## **Step 4: Extract CBAM data and compare with ground data**

In [ ]:
# @title Step 4: Load CBAM (or pull ERA5 equivalent from GEE) {"display-mode":"form"}
# @markdown CBAM (2018-2024) loaded from local directory if available.
# @markdown If not — pulls ERA5 Land min/max temperature from GEE, saves locally, and continues.

from itertools import combinations_with_replacement
import xarray as xr, numpy as np, pandas as pd, os, glob
from IPython.display import display, HTML

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — reads from the notebook's top config cell (Country / Country_region)
# No need to redefine country or bbox here
# ─────────────────────────────────────────────────────────────────────────────
# GEE_PROJECT = "natural-notch-435413-j3"

# Read from the config cell variables
COUNTRY = Country                              # "Ethiopia", "Tanzania", etc.
xmin, ymin, xmax, ymax = Country_region[COUNTRY]

print(f'  Country : {COUNTRY}')
print(f'  Bounds  : lon [{xmin}, {xmax}]  lat [{ymin}, {ymax}]')

# Date window — first 5 days of March 2024
CBAM_START = '2024-03-01'
CBAM_END   = '2024-03-05'

# Local paths
cbam_shared_path = './data/CBAM_temp2018_2024.nc'
era5_base_path = './data/era5'
os.makedirs(era5_base_path, exist_ok=True)

# Check if CBAM file exists, if not, check current directory or prompt to upload
if not os.path.exists(cbam_shared_path):
    if os.path.exists('./CBAM_temp2018_2024.nc'):
        os.makedirs('./data', exist_ok=True)
        import shutil
        shutil.move('./CBAM_temp2018_2024.nc', cbam_shared_path)
        print(f'   ✅ Moved CBAM file to {cbam_shared_path}')
    else:
        print(f'⚠️  CBAM file not found. Please upload CBAM_temp2018_2024.nc:')
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.nc'):
                os.makedirs('./data', exist_ok=True)
                os.rename(fname, cbam_shared_path)
                print(f'   ✅ Saved as {cbam_shared_path}')
                break

era5_country_tmin = os.path.join(era5_base_path, f'era5_tmin_march_2024_{COUNTRY}.nc')
era5_country_tmax = os.path.join(era5_base_path, f'era5_tmax_march_2024_{COUNTRY}.nc')

# ─────────────────────────────────────────────────────────────────────────────
# Everything below this line is unchanged from the previous version
# ─────────────────────────────────────────────────────────────────────────────

def subset_region(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower()), 'lat')
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower()), 'lon')
    lat_vals = ds[lat_dim].values
    lat_slice = (slice(ymax, ymin) if lat_vals[0] > lat_vals[-1]
                 else slice(ymin, ymax))
    return ds.sel({lat_dim: lat_slice, lon_dim: slice(xmin, xmax)})

def pull_era5_temp_var(gee_band, label, start, end, bbox, out_nc):
    import ee
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
    try:
        import geemap
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'geemap', '-q'], check=True)
        import geemap
    from tqdm.notebook import tqdm
    import rasterio as rio

    lon_min, lat_min, lon_max, lat_max = bbox
    roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
    col = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
          .filterDate(start, end)
          .filterBounds(roi)
          .select(gee_band)
          .map(lambda img: img
               .subtract(273.15)
               .rename(gee_band)
               .copyProperties(img, ['system:time_start']))
    )
    n = col.size().getInfo()
    print(f'   GEE: {label} — {n} daily images ({start} → {end})')

    tmp_dir = f'./tmp/era5_{label}_{COUNTRY}'
    os.makedirs(tmp_dir, exist_ok=True)
    img_list = col.toList(n)

    for i in tqdm(range(n), desc=f'{label} ({COUNTRY})', unit='day'):
        img      = ee.Image(img_list.get(i))
        date_str = img.date().format('yyyyMMdd').getInfo()
        out_file = os.path.join(tmp_dir, f'{date_str}.tif')
        if os.path.exists(out_file):
            continue
        geemap.ee_export_image(img, filename=out_file,
                               scale=11132, region=roi, file_per_band=False)

    tiffs = sorted(glob.glob(os.path.join(tmp_dir, '*.tif')))
    print(f'   Assembling {len(tiffs)} TIFFs → NetCDF...')
    if not tiffs:
        print(f'   ❌ No TIFFs found for {label}')
        return None

    with rio.open(tiffs[0]) as src:
        transform     = src.transform
        height, width = src.height, src.width

    lons = np.array([transform.c + (i + 0.5) * transform.a for i in range(width)])
    lats = np.array([transform.f + (j + 0.5) * transform.e for j in range(height)])
    stack = np.zeros((len(tiffs), height, width), dtype=np.float32)
    for i, tif in enumerate(tiffs):
        with rio.open(tif) as src:
            stack[i] = src.read(1).astype(np.float32)

    dates = pd.to_datetime(
        [os.path.basename(f).split('.')[0] for f in tiffs], format='%Y%m%d'
    )
    ds = xr.Dataset(
        {gee_band: (['time', 'lat', 'lon'], stack)},
        coords={'time': dates, 'lat': lats, 'lon': lons}
    )
    ds = ds.where(ds > -200)
    ds.to_netcdf(out_nc)
    print(f'   ✅ Saved: {out_nc}')
    return ds

# ── Attempt 1: CBAM ──────────────────────────────────────────────────────────
cbam_data  = None
data_label = None
grid_var   = None

print(f'\n── Temperature grid for {COUNTRY} ──────────────────────────────────')

if os.path.exists(cbam_shared_path):
    print(f'  CBAM found — subsetting to {COUNTRY}...')
    try:
        cbam_eac = xr.open_dataset(cbam_shared_path)
        time_dim = next((c for c in cbam_eac.coords if c in ['time','date']), None)
        if time_dim is None:
            raise ValueError('No time/date dimension in CBAM')
        cbam_data = (cbam_eac
                     .sel({time_dim: slice(CBAM_START, CBAM_END)})
                     .pipe(subset_region, xmin, xmax, ymin, ymax))
        del cbam_eac
        if 'date' in cbam_data.coords:
            cbam_data = cbam_data.rename({'date': 'time'})
        lat_dim = next(c for c in cbam_data.coords if 'lat' in c.lower())
        lon_dim = next(c for c in cbam_data.coords if 'lon' in c.lower())
        if cbam_data[lat_dim].size == 0 or cbam_data[lon_dim].size == 0:
            print(f'  ⚠️  CBAM empty for {COUNTRY} — likely outside coverage.')
            cbam_data = None
        else:
            if ('max_temperature' in cbam_data.data_vars and
                    'min_temperature' in cbam_data.data_vars):
                cbam_data['avg_temperature'] = (
                    (cbam_data['max_temperature'] +
                     cbam_data['min_temperature']) / 2
                )
                grid_var = 'max_temperature'
            else:
                grid_var = list(cbam_data.data_vars)[0]
            data_label = 'CBAM'
            print(f'  ✅ CBAM: {len(cbam_data.time)} days | variable: {grid_var}')
    except Exception as e:
        print(f'  ⚠️  CBAM load failed: {e}')
        cbam_data = None
else:
    print(f'  ⚠️  CBAM not available — CBAM covers EAC core countries only.')

# ── Attempt 2: ERA5 from GEE ─────────────────────────────────────────────────
if cbam_data is None:
    print(f'\n  Pulling ERA5 Land temperature from GEE for {COUNTRY}...')
    bbox = [xmin, ymin, xmax, ymax]

    ds_tmin = (xr.open_dataset(era5_country_tmin)
               if os.path.exists(era5_country_tmin)
               else pull_era5_temp_var('temperature_2m_min', 'tmin',
                                        CBAM_START, CBAM_END, bbox,
                                        era5_country_tmin))

    ds_tmax = (xr.open_dataset(era5_country_tmax)
               if os.path.exists(era5_country_tmax)
               else pull_era5_temp_var('temperature_2m_max', 'tmax',
                                        CBAM_START, CBAM_END, bbox,
                                        era5_country_tmax))

    if ds_tmin is not None and ds_tmax is not None:
        cbam_data = xr.Dataset({
            'min_temperature': ds_tmin['temperature_2m_min'],
            'max_temperature': ds_tmax['temperature_2m_max'],
        })
        cbam_data['avg_temperature'] = (
            (cbam_data['max_temperature'] + cbam_data['min_temperature']) / 2
        )
        grid_var   = 'max_temperature'
        data_label = 'ERA5 Land'
        print(f'  ✅ ERA5 Land ready as CBAM substitute ({len(cbam_data.time)} days)')
    else:
        print('  ❌ GEE pull failed.')

# ── Summary ───────────────────────────────────────────────────────────────────
if cbam_data is not None:
    t_vals = cbam_data[grid_var].values
    print(f'\n{"─"*60}')
    print(f'  Source   : {data_label}')
    print(f'  Variable : {grid_var}')
    print(f'  Period   : {CBAM_START} → {CBAM_END}')
    print(f'  Temp     : {np.nanmin(t_vals):.1f}°C → {np.nanmax(t_vals):.1f}°C')
    print(f'{"─"*60}')

# ── Guards + animation ────────────────────────────────────────────────────────
_tahmo_ok = 'tahmo_tmax' in dir() and tahmo_tmax is not None and not tahmo_tmax.empty
_info_ok  = 'info' in dir() and info is not None and not info.empty

if cbam_data is None:
    print('\n⚠️  No temperature grid available.')
elif not _tahmo_ok:
    print(f'\n⚠️  tahmo_tmax not found — run Steps 2b/2c first.')
    print(f'   {data_label} grid is ready.')
elif not _info_ok:
    print('\n⚠️  Station metadata (info) not found — run the metadata step first.')
else:
    grid_times   = cbam_data.time.values
    common_times = tahmo_tmax.index.intersection(grid_times)

    if len(common_times) == 0:
        print(f'\n⚠️  No common dates between TAHMO and {data_label}.')
    else:
        _id_col  = next((c for c in info.columns
                         if c.lower() in ['code','station_id','id']), None)
        _lat_col = next((c for c in info.columns if 'lat' in c.lower()), None)
        _lon_col = next((c for c in info.columns
                         if 'lon' in c.lower() or 'lng' in c.lower()), None)

        if not all([_id_col, _lat_col, _lon_col]):
            print(f'⚠️  Cannot detect id/lat/lon in info: {list(info.columns)}')
        else:
            common_stations   = [s for s in tahmo_tmax.columns
                                if s in info[_id_col].values]
            info_filtered     = info[info[_id_col].isin(common_stations)].copy()
            tahmo_tmax_sliced = tahmo_tmax.loc[common_times, common_stations]

            # ── Force lat/lon to numeric — CSV often loads them as strings ────────
            info_filtered[_lat_col] = pd.to_numeric(info_filtered[_lat_col],
                                                      errors='coerce')
            info_filtered[_lon_col] = pd.to_numeric(info_filtered[_lon_col],
                                                      errors='coerce')
            info_filtered = info_filtered.dropna(subset=[_lat_col, _lon_col])

            # ── Also force tahmo_tmax values to numeric ───────────────────────────
            tahmo_tmax_sliced = tahmo_tmax_sliced.apply(
                pd.to_numeric, errors='coerce'
            )

            # ── Re-align stations after dropping any with bad coordinates ─────────
            valid_stations    = info_filtered[_id_col].tolist()
            tahmo_tmax_sliced = tahmo_tmax_sliced[
                [s for s in tahmo_tmax_sliced.columns if s in valid_stations]
            ]

            print(f'  {len(common_times)} timesteps | '
                  f'{len(valid_stations)} stations with valid coordinates')

            if tahmo_tmax_sliced.empty or info_filtered.empty:
                print('⚠️  No valid station data after coordinate cleanup.')
            else:
                html_anim = point_plot(
                    tahmo_tmax_sliced,
                    info_filtered,
                    variable_name='Ground Tmax (°C)',
                    metadata_columns=[_id_col, _lat_col, _lon_col],
                    cmap='plasma',
                    grid_da=cbam_data.sel(time=common_times),
                    grid_cmap='coolwarm',
                    grid_alpha=0.5,
                    fig_title=(f'{COUNTRY} — Station Tmax vs {data_label} '
                              f'({CBAM_START} → {CBAM_END})'),
                    grid_da_var=grid_var,
                )
                print(f'✅ Animation: Station Tmax vs {data_label}')
                display(html_anim)

## **Step 5: Compare CBAM and ERA5**

In [ ]:
# @title CBAM vs ERA5 Comparison (or ERA5-only for new countries)
# @markdown Compares tmin and tmax between CBAM and ERA5 where CBAM is available.
# @markdown For Ethiopia / Tanzania / Eritrea / Djibouti — ERA5 tmin vs tmax comparison only.

import math, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib import animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import HTML, display
import xarray as xr

# ── Read config from top cell ─────────────────────────────────────────────────
COUNTRY = Country
xmin, ymin, xmax, ymax = Country_region[COUNTRY]

# ── Date window (must match Step 4) ──────────────────────────────────────────
start_date_slice = '2024-03-01'
end_date_slice   = '2024-03-05'

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: derive imshow extent from an xarray DataArray
# Handles lat/lon and y/x coordinate names — no utils import needed
# ─────────────────────────────────────────────────────────────────────────────
def get_extent_from_xarray(da):
    if 'lon' in da.coords and 'lat' in da.coords:
        x, y = da['lon'].values, da['lat'].values
    elif 'x' in da.coords and 'y' in da.coords:
        x, y = da['x'].values, da['y'].values
    elif 'longitude' in da.coords and 'latitude' in da.coords:
        x, y = da['longitude'].values, da['latitude'].values
    else:
        raise ValueError(f'No spatial coords found. Available: {list(da.coords)}')

    dx = (x[1] - x[0]) / 2 if len(x) > 1 else 0.0
    dy = abs(y[1] - y[0]) / 2 if len(y) > 1 else 0.0
    lon_min, lon_max = x[0] - dx, x[-1] + dx
    if y[0] < y[-1]:
        lat_min, lat_max = y[0] - dy, y[-1] + dy
    else:
        lat_min, lat_max = y[-1] - dy, y[0] + dy

    return [lon_min, lon_max, lat_min, lat_max]


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: animated multi-panel plot
# ─────────────────────────────────────────────────────────────────────────────
def plot_multiple_data(data_dict, fig_title, plot_size=5, robust=False,
                       cols=2, polygon=None):
    num_plots = len(data_dict)
    cols      = min(cols, num_plots)
    rows      = math.ceil(num_plots / cols)

    fig = plt.figure(figsize=(plot_size * 2 * cols, plot_size * rows),
                     constrained_layout=False)
    fig.subplots_adjust(left=0, right=0.75, top=0.93, bottom=0.02,
                        wspace=0.05, hspace=0.15)

    if fig_title:
        fig.suptitle(fig_title, fontsize=12, y=0.98)

    images, axes, precomputed = [], [], {}
    max_steps = 1

    for idx, (title, data_tuple) in enumerate(data_dict.items()):
        data, norm, cmap = data_tuple

        ax = fig.add_subplot(rows, cols, idx + 1,
                             projection=ccrs.PlateCarree())
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(title, fontsize=9)
        ax.add_feature(cfeature.COASTLINE,  linewidth=0.6)
        ax.add_feature(cfeature.BORDERS,    linestyle=':', linewidth=0.5)
        ax.add_feature(cfeature.LAND,       facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN,      facecolor='aliceblue')

        if polygon is not None:
            ax.add_patch(mpatches.Polygon(
                polygon, closed=True, facecolor='none',
                edgecolor='red', linewidth=1.5,
                transform=ccrs.PlateCarree()
            ))

        time_steps = data.sizes.get('time', 1)
        max_steps  = max(max_steps, time_steps)
        precomputed[title] = [
            data.isel(time=t, missing_dims='ignore') if time_steps > 1
            else data
            for t in range(time_steps)
        ]

        extent = get_extent_from_xarray(precomputed[title][0])
        im = ax.imshow(
            precomputed[title][0].values,
            norm=norm, cmap=cmap,
            origin='lower',
            transform=ccrs.PlateCarree(),
            extent=extent
        )
        cbar = plt.colorbar(im, ax=ax, orientation='vertical',
                            pad=0.03, aspect=18, shrink=0.75,
                            extend='both' if robust else 'neither')
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label('°C', fontsize=8)

        images.append(im)
        axes.append(ax)

    def update(frame):
        for idx, (title, (data, _, _)) in enumerate(data_dict.items()):
            t = min(frame, data.sizes.get('time', 1) - 1)
            images[idx].set_data(precomputed[title][t].values)
        return images

    ani = animation.FuncAnimation(fig, update, frames=max_steps,
                                  interval=300, blit=True)
    plt.close(fig)
    return ani, HTML(ani.to_jshtml())


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: compare a list of single-variable datasets with a shared colour scale
# ─────────────────────────────────────────────────────────────────────────────
def compare_xarray_datasets(datasets, labels, fig_title,
                             plot_size=4, robust=True, cols=2,
                             polygon=None, cmap='coolwarm', save=False):
    all_values = []
    for ds in datasets:
        var = list(ds.data_vars)[0]
        vals = ds[var].values.flatten()
        all_values.extend(vals[np.isfinite(vals)])

    vmin, vmax = np.nanmin(all_values), np.nanmax(all_values)
    if robust:
        vmin = np.nanpercentile(all_values, 2)
        vmax = np.nanpercentile(all_values, 98)

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    print(f'  Shared colour scale: {vmin:.1f}°C → {vmax:.1f}°C')

    data_dict = {}
    for ds, label in zip(datasets, labels):
        var = list(ds.data_vars)[0]
        data_dict[label] = (ds[var], norm, cmap)

    ani, html_anim = plot_multiple_data(
        data_dict, fig_title,
        plot_size=plot_size, robust=robust,
        cols=cols, polygon=polygon
    )
    if save:
        ani.save(f'{fig_title.replace(" ","_")}.gif',
                 writer='pillow', fps=3, dpi=150,
                 savefig_kwargs={'facecolor': 'white'})
    return html_anim


# ─────────────────────────────────────────────────────────────────────────────
# DECIDE WHAT TO COMPARE
# cbam_data comes from Step 4 — it's either real CBAM or ERA5 masquerading as CBAM
# era5_tmin / era5_tmax come from Step 5
# ─────────────────────────────────────────────────────────────────────────────
_has_cbam  = (cbam_data is not None and
              data_label == 'CBAM' and
              'min_temperature' in cbam_data.data_vars and
              'max_temperature' in cbam_data.data_vars)

_has_era5  = ('era5_tmin' in dir() and era5_tmin is not None and
              'era5_tmax' in dir() and era5_tmax is not None)

print(f'  Country      : {COUNTRY}')
print(f'  CBAM on Drive: {"✅ yes" if _has_cbam else "⚠️  no"}')
print(f'  ERA5 loaded  : {"✅ yes" if _has_era5 else "⚠️  no"}')

# ─────────────────────────────────────────────────────────────────────────────
# CASE 1: both CBAM and ERA5 available → four-panel comparison (original intent)
# ─────────────────────────────────────────────────────────────────────────────
if _has_cbam and _has_era5:
    print('\n  Mode: 4-panel CBAM vs ERA5 comparison')

    cbam_tmin_sliced = (cbam_data['min_temperature']
                        .to_dataset(name='min_temperature')
                        .sel(time=slice(start_date_slice, end_date_slice)))
    cbam_tmax_sliced = (cbam_data['max_temperature']
                        .to_dataset(name='max_temperature')
                        .sel(time=slice(start_date_slice, end_date_slice)))
    era5_tmin_sliced = era5_tmin.sel(time=slice(start_date_slice, end_date_slice))
    era5_tmax_sliced = era5_tmax.sel(time=slice(start_date_slice, end_date_slice))

    html = compare_xarray_datasets(
        datasets=[era5_tmin_sliced, era5_tmax_sliced,
                  cbam_tmin_sliced, cbam_tmax_sliced],
        labels=['ERA5 Tmin', 'ERA5 Tmax',
                'CBAM Tmin', 'CBAM Tmax'],
        fig_title=f'{COUNTRY} — ERA5 vs CBAM Temperature ({start_date_slice} → {end_date_slice})',
        plot_size=3, robust=True, cols=2, cmap='coolwarm'
    )
    print('✅ 4-panel comparison generated')
    display(html)

# ─────────────────────────────────────────────────────────────────────────────
# CASE 2: ERA5 only (new countries) → two-panel ERA5 tmin vs tmax
# Uses cbam_data which was populated with ERA5 values in Step 4
# ─────────────────────────────────────────────────────────────────────────────
elif cbam_data is not None and not _has_cbam:
    print(f'\n  Mode: 2-panel ERA5 Tmin vs Tmax '
          f'(CBAM not available for {COUNTRY})')

    era5_tmin_sliced = (cbam_data['min_temperature']
                        .to_dataset(name='min_temperature')
                        .sel(time=slice(start_date_slice, end_date_slice)))
    era5_tmax_sliced = (cbam_data['max_temperature']
                        .to_dataset(name='max_temperature')
                        .sel(time=slice(start_date_slice, end_date_slice)))

    html = compare_xarray_datasets(
        datasets=[era5_tmin_sliced, era5_tmax_sliced],
        labels=['ERA5 Land Tmin (°C)', 'ERA5 Land Tmax (°C)'],
        fig_title=f'{COUNTRY} — ERA5 Land Temperature ({start_date_slice} → {end_date_slice})',
        plot_size=4, robust=True, cols=2, cmap='coolwarm'
    )
    print('✅ ERA5 comparison generated')
    display(html)

    # ── Bonus: diurnal range map ──────────────────────────────────────────────
    print('\n  Computing diurnal temperature range (Tmax − Tmin)...')
    dtr = (cbam_data['max_temperature'] -
           cbam_data['min_temperature']).mean(dim='time')
    dtr_ds = dtr.to_dataset(name='diurnal_range')

    html_dtr = compare_xarray_datasets(
        datasets=[dtr_ds],
        labels=[f'{COUNTRY} — Mean diurnal range (°C)'],
        fig_title='',
        plot_size=5, robust=False, cols=1, cmap='YlOrRd'
    )
    print('✅ Diurnal range map generated')
    display(html_dtr)

# ─────────────────────────────────────────────────────────────────────────────
# CASE 3: nothing loaded
# ─────────────────────────────────────────────────────────────────────────────
else:
    print('\n⚠️  No temperature data available.')
    print('   Run Step 4 to pull ERA5 temperature from GEE first.')

## **Step 6: Compute PET and stress days with CBAM and ERA5**

In [ ]:
# @title ###  6a) Hargreaves Equation for Potential Evapotranspiration (PET)
# @markdown The Hargreaves method estimates daily potential evapotranspiration (PET)
# @markdown based on temperature range and incoming solar radiation:
# @markdown
# @markdown $$
# @markdown \text{PET} = 0.0023 \times R_a \times (T_{mean} + 17.8) \times \sqrt{T_{max} - T_{min}}
# @markdown $$
# @markdown
# @markdown **Where:**
# @markdown - $PET$: Potential Evapotranspiration (mm day⁻¹)
# @markdown - $R_a$: Extraterrestrial radiation (MJ m⁻² day¹)
# @markdown - $T_{mean}$: Mean daily air temperature (°C)
# @markdown - $T_{max}$: Maximum daily air temperature (°C)
# @markdown - $T_{min}$: Minimum daily air temperature (°C)
# @markdown
# @markdown 💡 *In this function, a default value of $R_a = 15.0$ MJ m⁻² day⁻¹ is used* <br>

# @markdown ### **Stress Condition Criteria**
# @markdown A grid cell or station is considered under **heat stress** when:
# @markdown
# @markdown - $PET > 5$ mm day⁻¹  <br>
# @markdown **and**
# @markdown - Maximum temperature ($T_{max} > 32\,°C$)

# @title Step 6a: Hargreaves PET and Heat Stress
# @markdown Computes PET using the Hargreaves equation for available temperature data.
# @markdown CBAM used where available; ERA5 Land used for Ethiopia/Tanzania/Eritrea/Djibouti.
# @markdown
# @markdown $$\text{PET} = 0.0023 \times R_a \times (T_{mean} + 17.8) \times \sqrt{T_{max} - T_{min}}$$
# @markdown
# @markdown **Stress criteria:** PET > 5 mm day⁻¹  AND  Tmax > 32 °C

import xarray as xr, numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import display

# ── Config from top cell ──────────────────────────────────────────────────────
COUNTRY = Country
xmin, ymin, xmax, ymax = Country_region[COUNTRY]

MARCH_START = '2024-03-01'
MARCH_END   = '2024-03-31'

# ─────────────────────────────────────────────────────────────────────────────
# HARGREAVES PET
# ─────────────────────────────────────────────────────────────────────────────
def pet_hargreaves(tmin, tmax, tmean, Ra=15.0):
    """PET in mm/day. Ra = 15.0 MJ m⁻² day¹ (workshop constant). All °C."""
    dtr = np.maximum(tmax - tmin, 0)
    return 0.0023 * Ra * (tmean + 17.8) * np.sqrt(dtr)

def rmse(a, b):
    return float(np.sqrt(np.nanmean((np.asarray(a) - np.asarray(b)) ** 2)))

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: safe spatial subset
# ─────────────────────────────────────────────────────────────────────────────
def subset_region(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower()), None)
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower()), None)
    if lat_dim is None or lon_dim is None:
        return ds
    lat_vals  = ds[lat_dim].values
    lat_slice = (slice(ymax, ymin) if lat_vals[0] > lat_vals[-1]
                 else slice(ymin, ymax))
    return ds.sel({lat_dim: lat_slice, lon_dim: slice(xmin, xmax)})

# ─────────────────────────────────────────────────────────────────────────────
# LOAD CBAM (where available — Kenya/Uganda/Rwanda only)
# ─────────────────────────────────────────────────────────────────────────────
cbam_shared_path = './data/temperature/reanalysis/CBAM_temp2018_2024.nc'
os.makedirs(os.path.dirname(cbam_shared_path), exist_ok=True)

_has_cbam_file = False
cbam_march     = None

print(f'  Country : {COUNTRY}')
print(f'  Period  : {MARCH_START} → {MARCH_END}\n')

if os.path.exists(cbam_shared_path):
    print('  Loading CBAM for March 2024...')
    try:
        _raw = xr.open_dataset(cbam_shared_path)
        _td  = next((c for c in _raw.coords if c in ['time', 'date']), None)
        if _td:
            cbam_march = (_raw
                          .sel({_td: slice(MARCH_START, MARCH_END)})
                          .pipe(subset_region, xmin, xmax, ymin, ymax))
            del _raw
            if 'date' in cbam_march.coords:
                cbam_march = cbam_march.rename({'date': 'time'})
            _lat = next(c for c in cbam_march.coords if 'lat' in c.lower())
            if cbam_march[_lat].size > 0:
                cbam_march['avg_temperature'] = (
                    (cbam_march['max_temperature'] +
                     cbam_march['min_temperature']) / 2
                )
                _has_cbam_file = True
                print(f'  ✅ CBAM loaded: {len(cbam_march.time)} days')
            else:
                print(f'  ⚠️  CBAM empty for {COUNTRY} bbox — ERA5 only')
                cbam_march = None
    except Exception as e:
        print(f'  ⚠️  CBAM load error: {e}')
        cbam_march = None
else:
    print(f'  ℹ️  CBAM not available locally — ERA5-only mode for {COUNTRY}')

# ─────────────────────────────────────────────────────────────────────────────
# LOAD ERA5 FULL MARCH
# Uses separate *_full_* filenames so Step 4's 5-day files are never touched.
# Explicit .close() before any delete prevents PermissionError.
# ─────────────────────────────────────────────────────────────────────────────
era5_base     = './data/temperature/reanalysis/era5'
os.makedirs(era5_base, exist_ok=True)
_e5_tmin_path = os.path.join(era5_base, f'era5_tmin_march_2024_full_{COUNTRY}.nc')
_e5_tmax_path = os.path.join(era5_base, f'era5_tmax_march_2024_full_{COUNTRY}.nc')

_era5_tmin_ds = None
_era5_tmax_ds = None

for _path, _tag in [(_e5_tmin_path, 'tmin'), (_e5_tmax_path, 'tmax')]:
    _ds = None
    if os.path.exists(_path):
        try:
            _ds    = xr.open_dataset(_path)
            n_days = len(_ds.time)
            if n_days >= 28:
                print(f'  ✅ ERA5 {_tag}: {n_days} days loaded from local cache')
            else:
                # Too short — close cleanly before deleting
                print(f'  ⚠️  ERA5 {_tag} only {n_days} days — re-pulling...')
                _ds.close()
                _ds = None
                os.remove(_path)
                print(f'     Removed stale file: {_path}')
        except Exception as e:
            print(f'  ⚠️  Could not open {_tag}: {e}')
            try:
                if _ds is not None:
                    _ds.close()
            except Exception:
                pass
            _ds = None

    if _tag == 'tmin':
        _era5_tmin_ds = _ds
    else:
        _era5_tmax_ds = _ds

# Pull from GEE for any variable still missing
_bbox = [xmin, ymin, xmax, ymax]

if _era5_tmin_ds is None:
    print('\n  Pulling ERA5 tmin for full March from GEE...')
    _era5_tmin_ds = pull_era5_temp_var(
        gee_band='temperature_2m_min', label='tmin_full',
        start=MARCH_START, end=MARCH_END,
        bbox=_bbox, out_nc=_e5_tmin_path
    )

if _era5_tmax_ds is None:
    print('\n  Pulling ERA5 tmax for full March from GEE...')
    _era5_tmax_ds = pull_era5_temp_var(
        gee_band='temperature_2m_max', label='tmax_full',
        start=MARCH_START, end=MARCH_END,
        bbox=_bbox, out_nc=_e5_tmax_path
    )

if _era5_tmin_ds is None or _era5_tmax_ds is None:
    raise RuntimeError('ERA5 temperature unavailable. '
                       'Check GEE access or local paths.')

_tmin_var = list(_era5_tmin_ds.data_vars)[0]
_tmax_var = list(_era5_tmax_ds.data_vars)[0]
print(f'\n  ERA5 variables : {_tmin_var}, {_tmax_var}')
print(f'  ERA5 tmin days : {len(_era5_tmin_ds.time)}')
print(f'  ERA5 tmax days : {len(_era5_tmax_ds.time)}')

era5_tmean = (_era5_tmin_ds[_tmin_var] + _era5_tmax_ds[_tmax_var]) / 2

# ─────────────────────────────────────────────────────────────────────────────
# COMPUTE PET
# ─────────────────────────────────────────────────────────────────────────────
print('\n  Computing PET (Hargreaves, Ra=15 MJ m⁻² day⁻¹)...')

pet_era5_da = pet_hargreaves(
    _era5_tmin_ds[_tmin_var],
    _era5_tmax_ds[_tmax_var],
    era5_tmean
)
pet_era5 = pet_era5_da.to_dataset(name='pet')
print(f'  ✅ ERA5 PET : mean={float(pet_era5_da.mean()):.2f}  '
      f'max={float(pet_era5_da.max()):.2f} mm/day')

pet_cbam = None
if _has_cbam_file and cbam_march is not None:
    pet_cbam_da = pet_hargreaves(
        cbam_march['min_temperature'],
        cbam_march['max_temperature'],
        cbam_march['avg_temperature']
    )
    pet_cbam = pet_cbam_da.to_dataset(name='pet')
    print(f'  ✅ CBAM PET : mean={float(pet_cbam_da.mean()):.2f}  '
          f'max={float(pet_cbam_da.max()):.2f} mm/day')

    _common = np.intersect1d(pet_era5.time.values, pet_cbam.time.values)
    if len(_common) > 0:
        _r = rmse(
            pet_era5['pet'].sel(time=_common).values.flatten(),
            pet_cbam['pet'].sel(time=_common).values.flatten()
        )
        print(f'  RMSE ERA5 vs CBAM PET : {_r:.3f} mm/day')

# ─────────────────────────────────────────────────────────────────────────────
# HEAT STRESS (PET > 5 mm/day AND Tmax > 32°C)
# ─────────────────────────────────────────────────────────────────────────────
print('\n  Computing heat stress (PET>5 AND Tmax>32°C)...')

stress_era5  = ((pet_era5['pet'] > 5) &
                (_era5_tmax_ds[_tmax_var] > 32)).astype(int).to_dataset(name='stress')
_n_era5      = int(stress_era5['stress'].sum())
_tot_era5    = stress_era5['stress'].size
_pct_era5    = _n_era5 / _tot_era5 * 100
print(f'  ERA5  stress: {_n_era5:,}/{_tot_era5:,} ({_pct_era5:.2f}%)')

stress_cbam = None
if pet_cbam is not None:
    stress_cbam  = ((pet_cbam['pet'] > 5) &
                    (cbam_march['max_temperature'] > 32)).astype(int).to_dataset(name='stress')
    _n_cbam      = int(stress_cbam['stress'].sum())
    _tot_cbam    = stress_cbam['stress'].size
    _pct_cbam    = _n_cbam / _tot_cbam * 100
    print(f'  CBAM  stress: {_n_cbam:,}/{_tot_cbam:,} ({_pct_cbam:.2f}%)')

# ─────────────────────────────────────────────────────────────────────────────
# VISUALISE: mean PET and cumulative stress days
# ─────────────────────────────────────────────────────────────────────────────
def _map_panel(ax, data_2d, lat, lon, title, cmap, vmin, vmax, unit):
    ax.add_feature(cfeature.COASTLINE,  linewidth=0.6)
    ax.add_feature(cfeature.BORDERS,    linestyle=':', linewidth=0.4)
    ax.add_feature(cfeature.LAND,       facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN,      facecolor='aliceblue')
    im = ax.pcolormesh(
        lon, lat, data_2d,
        cmap=cmap,
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax),
        transform=ccrs.PlateCarree()
    )
    plt.colorbar(im, ax=ax, orientation='vertical',
                 shrink=0.75, pad=0.03, label=unit)
    ax.set_title(title, fontsize=9)
    ax.set_extent([xmin, xmax, ymin, ymax], crs=ccrs.PlateCarree())

_lat_e = (_era5_tmax_ds['lat'].values if 'lat' in _era5_tmax_ds.coords
          else _era5_tmax_ds['y'].values)
_lon_e = (_era5_tmax_ds['lon'].values if 'lon' in _era5_tmax_ds.coords
          else _era5_tmax_ds['x'].values)

n_cols = 4 if pet_cbam is not None else 2
fig, axes = plt.subplots(
    1, n_cols, figsize=(5 * n_cols, 5),
    subplot_kw={'projection': ccrs.PlateCarree()}
)
if n_cols == 2:
    axes = list(axes)

_map_panel(
    axes[0],
    pet_era5['pet'].mean(dim='time').values,
    _lat_e, _lon_e,
    'ERA5 Mean PET\nMarch 2024',
    'YlOrRd', 0, float(pet_era5['pet'].max()), 'mm/day'
)
_map_panel(
    axes[1],
    stress_era5['stress'].sum(dim='time').values,
    _lat_e, _lon_e,
    'ERA5 Cumulative stress days\n(PET>5 & Tmax>32°C)',
    'Reds', 0, int(stress_era5['stress'].sum(dim='time').max()), 'days'
)

if pet_cbam is not None:
    _lat_c = cbam_march['lat'].values
    _lon_c = cbam_march['lon'].values
    _map_panel(
        axes[2],
        pet_cbam['pet'].mean(dim='time').values,
        _lat_c, _lon_c,
        'CBAM Mean PET\nMarch 2024',
        'YlOrRd', 0, float(pet_cbam['pet'].max()), 'mm/day'
    )
    _map_panel(
        axes[3],
        stress_cbam['stress'].sum(dim='time').values,
        _lat_c, _lon_c,
        'CBAM Cumulative stress days\n(PET>5 & Tmax>32°C)',
        'Reds', 0, int(stress_cbam['stress'].sum(dim='time').max()), 'days'
    )

fig.suptitle(
    f'{COUNTRY} — Hargreaves PET & Heat Stress | March 2024',
    fontsize=12, y=1.01
)
plt.tight_layout()

RESULTS_DIR = f'./results/SkillExplorer_{COUNTRY}_{pd.Timestamp.today().date()}'
os.makedirs(RESULTS_DIR, exist_ok=True)
out = os.path.join(RESULTS_DIR, f'pet_stress_{COUNTRY}.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  💾 Saved: {out}')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
rows = [{
    'source':          'ERA5 Land',
    'mean_PET_mm_day': round(float(pet_era5['pet'].mean()), 2),
    'max_PET_mm_day':  round(float(pet_era5['pet'].max()),  2),
    'stress_points':   _n_era5,
    'total_points':    _tot_era5,
    'stress_pct':      round(_pct_era5, 2)
}]

if pet_cbam is not None:
    rows.append({
        'source':          'CBAM',
        'mean_PET_mm_day': round(float(pet_cbam['pet'].mean()), 2),
        'max_PET_mm_day':  round(float(pet_cbam['pet'].max()),  2),
        'stress_points':   _n_cbam,
        'total_points':    _tot_cbam,
        'stress_pct':      round(_pct_cbam, 2)
    })

df_pet = pd.DataFrame(rows)
display(df_pet)
df_pet.to_csv(
    os.path.join(RESULTS_DIR, f'pet_stress_summary_{COUNTRY}.csv'),
    index=False
)
print(f'\n✅ Step 6a complete — {COUNTRY}')

In [ ]:
# @title Step 6b: Visualise Stress Days
# @markdown Shows a randomly selected stress day and a frequency summary.
# @markdown ERA5 used for Ethiopia/Tanzania/Eritrea/Djibouti; CBAM where available.

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import pandas as pd
import os

# ── Config from top cell ──────────────────────────────────────────────────────
COUNTRY = Country
xmin, ymin, xmax, ymax = Country_region[COUNTRY]

RESULTS_DIR = f'./results/SkillExplorer_{COUNTRY}_{pd.Timestamp.today().date()}'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: plot one stress map panel
# ─────────────────────────────────────────────────────────────────────────────
def _stress_panel(ax, data_2d, lat, lon, title, date_str, stress_count):
    """Plot a binary (0/1) stress map on a cartopy axes."""
    cmap_s = mcolors.ListedColormap(['lightblue', 'red'])
    norm_s = mcolors.BoundaryNorm([-0.5, 0.5, 1.5], cmap_s.N)

    ax.add_feature(cfeature.COASTLINE,  linewidth=0.8)
    ax.add_feature(cfeature.BORDERS,    linestyle=':', linewidth=0.5)
    ax.add_feature(cfeature.LAND,       facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN,      facecolor='aliceblue')
    ax.gridlines(draw_labels=True, linewidth=0.5,
                 color='gray', alpha=0.4, linestyle='--')
    ax.set_extent([xmin, xmax, ymin, ymax], crs=ccrs.PlateCarree())

    im = ax.pcolormesh(lon, lat, data_2d,
                       cmap=cmap_s, norm=norm_s,
                       transform=ccrs.PlateCarree())

    cbar = plt.colorbar(im, ax=ax, orientation='vertical',
                        pad=0.05, shrink=0.7, ticks=[0, 1])
    cbar.set_label('Stress (0=No  1=Yes)', fontsize=9)
    cbar.ax.set_yticklabels(['No', 'Yes'])

    ax.set_title(f'{title}\n{date_str} | stress pixels: {stress_count:,}',
                 fontsize=9)

def _get_stress_info(stress_ds):
    time_dim = next((d for d in stress_ds.dims if d in ['time', 'date']), None)
    lat_arr  = (stress_ds['lat'].values  if 'lat'  in stress_ds.coords else
                stress_ds['y'].values    if 'y'    in stress_ds.coords else None)
    lon_arr  = (stress_ds['lon'].values  if 'lon'  in stress_ds.coords else
                stress_ds['x'].values    if 'x'    in stress_ds.coords else None)
    return time_dim, lat_arr, lon_arr

# ─────────────────────────────────────────────────────────────────────────────
# DECIDE WHICH STRESS DATASETS TO PLOT
# ─────────────────────────────────────────────────────────────────────────────
_plot_sources = {}

if 'stress_era5' in dir() and stress_era5 is not None:
    _plot_sources['ERA5 Land'] = stress_era5

if 'stress_cbam' in dir() and stress_cbam is not None:
    _plot_sources['CBAM'] = stress_cbam

if not _plot_sources:
    raise RuntimeError('No stress datasets available. Run Step 6a first.')

print(f'  Sources to plot : {list(_plot_sources.keys())}')
print(f'  Country         : {COUNTRY}\n')

# ─────────────────────────────────────────────────────────────────────────────
# FIND A RANDOM DAY WITH STRESS PIXELS
# ─────────────────────────────────────────────────────────────────────────────
_primary_label  = 'CBAM' if 'CBAM' in _plot_sources else 'ERA5 Land'
_primary_stress = _plot_sources[_primary_label]
_time_dim, _, _ = _get_stress_info(_primary_stress)

_spatial_dims = [d for d in _primary_stress['stress'].dims if d != _time_dim]
_daily_counts = _primary_stress['stress'].sum(dim=_spatial_dims)
_stress_days = _daily_counts.where(_daily_counts > 0, drop=True)

if len(_stress_days) == 0:
    print(f'ℹ️  No stress days found for {_primary_label} in the selected period.')
else:
    _random_day = np.random.choice(_stress_days[_time_dim].values)
    _day_str    = pd.Timestamp(_random_day).strftime('%Y-%m-%d')

    print(f'  Randomly selected stress day : {_day_str}')
    print(f'  Primary source               : {_primary_label}')

    # ── Build figure (one panel per source) ──────────────────────────────────
    n_panels = len(_plot_sources)
    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(7 * n_panels, 6),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    if n_panels == 1:
        axes = [axes]

    for ax, (label, stress_ds) in zip(axes, _plot_sources.items()):
        _td, _lat, _lon = _get_stress_info(stress_ds)
        if _lat is None or _lon is None:
            ax.set_title(f'{label}: no spatial coords found')
            continue

        try:
            _day_data = stress_ds['stress'].sel({_td: _random_day}, method='nearest').values
        except Exception as e:
            print(f'  ⚠️  {label}: could not select day — {e}')
            continue

        _count = int(np.nansum(_day_data))
        _stress_panel(ax, _day_data, _lat, _lon, label, _day_str, _count)

    fig.suptitle(
        f'{COUNTRY} — Heat Stress (PET>5 mm/day & Tmax>32°C)\n{_day_str}',
        fontsize=13, y=1.02
    )
    plt.tight_layout()

    out = os.path.join(RESULTS_DIR, f'stress_day_{_day_str}_{COUNTRY}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n  💾 Saved map: {out}')

    # ── Stress frequency bar chart (from the deleted 6b) ─────────────────────
    print(f'\n  Stress frequency summary — {COUNTRY}  (March 2024)')
    print(f'  {"─"*50}')

    fig2, ax2 = plt.subplots(figsize=(14, 4))
    for label, stress_ds in _plot_sources.items():
        _td, _, _ = _get_stress_info(stress_ds)
        _sp       = [d for d in stress_ds['stress'].dims if d != _td]
        _dc       = stress_ds['stress'].sum(dim=_sp)
        _days_hit = int((_dc > 0).sum())
        _total_d  = int(len(_dc))
        _max_px   = int(_dc.max())
        print(f'  {label:<12} : {_days_hit}/{_total_d} days had ≥1 stress pixel '
              f'| peak day = {_max_px:,} pixels')

        # Plot bar chart
        ax2.bar(
            pd.to_datetime(stress_ds[_td].values),
            _dc.values,
            alpha=0.7, width=1.5, label=label
        )

    ax2.axvline(pd.Timestamp(_random_day), color='black', linestyle='--', linewidth=1, label=f'Selected: {_day_str}')
    ax2.set_title(f'{COUNTRY} — Daily stress pixel count (March 2024)', fontsize=11)
    ax2.set_ylabel('Stressed pixels')
    ax2.set_xlabel('Date')
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    out2 = os.path.join(RESULTS_DIR, f'stress_frequency_{COUNTRY}.png')
    plt.savefig(out2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  💾 Saved chart: {out2}')
    print(f'  {"─"*50}')
    print('\n  💡 Re-run this cell to pick a different random stress day.')

## **Step 7: Visualise the heat change over the last half a century**

In [ ]:
# @title Step 7: ERA5 long-term monthly temperature (1982-2024)
# @markdown Loads pre-computed file from local directory if available.
# @markdown If not — pulls monthly ERA5 from GEE year by year and saves locally.
# @markdown Then plots a Month × Year heatmap for the selected country.

import xarray as xr, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, glob

# ── Config from top cell ──────────────────────────────────────────────────────
COUNTRY = Country
xmin, ymin, xmax, ymax = Country_region[COUNTRY]
# GEE_PROJECT = "natural-notch-435413-j3"

LT_START = 2020
LT_END   = 2024

# Local paths
_base_dir = './data/temperature/reanalysis/era5'
os.makedirs(_base_dir, exist_ok=True)

_shared_path  = os.path.join(_base_dir, 'era5_temperature_monthly_1982_2024_combined.nc')
_country_path = os.path.join(_base_dir, f'era5_temperature_monthly_{LT_START}_{LT_END}_{COUNTRY}.nc')

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: spatial subset (handles ascending/descending lat)
# ─────────────────────────────────────────────────────────────────────────────
def _subset(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower() or c == 'y'), None)
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower() or c == 'x'), None)
    if lat_dim is None or lon_dim is None:
        return ds
    lv = ds[lat_dim].values
    ls = slice(ymax, ymin) if lv[0] > lv[-1] else slice(ymin, ymax)
    return ds.sel({lat_dim: ls, lon_dim: slice(xmin, xmax)})

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: pull one year of monthly ERA5 temperature from GEE
# ─────────────────────────────────────────────────────────────────────────────
def _pull_era5_monthly_year(year, bbox, gee_project):
    import ee, rasterio as rio
    try:
        ee.Initialize(project=gee_project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=gee_project)
    try:
        import geemap
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'geemap', '-q'], check=True)
        import geemap

    lon_min, lat_min, lon_max, lat_max = bbox
    roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

    tmp_dir = f'./tmp/era5_lt_tmp/{year}'
    os.makedirs(tmp_dir, exist_ok=True)

    monthly_arrays, monthly_dates = [], []

    for month in range(1, 13):
        start = f'{year}-{month:02d}-01'
        end   = (pd.Timestamp(start) + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
        dstr  = f'{year}{month:02d}'
        out_f = os.path.join(tmp_dir, f'{dstr}.tif')

        if not os.path.exists(out_f):
            monthly_img = (
                ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
                  .filterDate(start, end)
                  .filterBounds(roi)
                  .select('temperature_2m')
                  .mean()
                  .subtract(273.15)
                  .clip(roi)
            )
            try:
                geemap.ee_export_image(
                    monthly_img, filename=out_f,
                    scale=11132, region=roi, file_per_band=False
                )
            except Exception as e:
                print(f'    ⚠️  {dstr}: {e}')
                continue

        if os.path.exists(out_f):
            with rio.open(out_f) as src:
                if month == 1:
                    transform     = src.transform
                    height, width = src.height, src.width
                arr = src.read(1).astype(np.float32)
            arr[arr < -200] = np.nan
            monthly_arrays.append(arr)
            monthly_dates.append(pd.Timestamp(f'{year}-{month:02d}-01'))

    if not monthly_arrays:
        return None

    with rio.open(os.path.join(tmp_dir, f'{year}{1:02d}.tif')) as src:
        t = src.transform
        h, w = src.height, src.width

    lons = np.array([t.c + (i + 0.5) * t.a for i in range(w)])
    lats = np.array([t.f + (j + 0.5) * t.e for j in range(h)])

    stack = np.stack(monthly_arrays, axis=0)
    return xr.Dataset(
        {'temperature_2m': (['time', 'lat', 'lon'], stack)},
        coords={'time': monthly_dates, 'lat': lats, 'lon': lons}
    )

# ─────────────────────────────────────────────────────────────────────────────
# LOAD STRATEGY
# ─────────────────────────────────────────────────────────────────────────────
era5_long_term = None

if os.path.exists(_country_path):
    print(f'✅ Loading country-specific file:\n   {_country_path}')
    era5_long_term = xr.open_dataset(_country_path)

elif os.path.exists(_shared_path):
    print(f'  Shared EAC file found — subsetting to {COUNTRY}...')
    try:
        _raw           = xr.open_dataset(_shared_path)
        era5_long_term = _subset(_raw, xmin, xmax, ymin, ymax)
        del _raw

        _var = list(era5_long_term.data_vars)[0]
        if era5_long_term[_var].max() > 200:
            era5_long_term[_var] = era5_long_term[_var] - 273.15
            print('  K → °C conversion applied')

        era5_long_term.to_netcdf(_country_path)
        print(f'  ✅ Subset saved for future runs:\n     {_country_path}')
    except Exception as e:
        print(f'  ⚠️  Shared file load failed: {e}')
        era5_long_term = None

else:
    print(f'  No long-term ERA5 file found for {COUNTRY}.')
    print(f'  Pulling monthly ERA5 {LT_START}–{LT_END} from GEE...')
    print(f'  ⏱  ~{(LT_END - LT_START + 1) * 2} min total  '
          f'(cached per year — restarts pick up where they left off)\n')

    bbox       = [xmin, ymin, xmax, ymax]
    year_parts = []

    for year in range(LT_START, LT_END + 1):
        print(f'  Year {year}...', end=' ', flush=True)
        ds_yr = _pull_era5_monthly_year(year, bbox, GEE_PROJECT)
        if ds_yr is not None:
            year_parts.append(ds_yr)
            print(f'✅ {len(ds_yr.time)} months')
        else:
            print('⚠️  skipped')

    if year_parts:
        era5_long_term = xr.concat(year_parts, dim='time')
        os.makedirs(os.path.dirname(_country_path), exist_ok=True)
        era5_long_term.to_netcdf(_country_path)
        print(f'\n  ✅ Saved locally: {_country_path}')
    else:
        print('\n  ❌ No data pulled. Check GEE access.')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY & PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
if era5_long_term is None:
    print('⚠️  ERA5 long-term data not available. Check options above.')
else:
    _var = list(era5_long_term.data_vars)[0]

    if float(era5_long_term[_var].max()) > 200:
        era5_long_term[_var] = era5_long_term[_var] - 273.15

    _rename = {}
    if 'y' in era5_long_term.coords and 'lat' not in era5_long_term.coords:
        _rename['y'] = 'lat'
    if 'x' in era5_long_term.coords and 'lon' not in era5_long_term.coords:
        _rename['x'] = 'lon'
    if _rename:
        era5_long_term = era5_long_term.rename(_rename)

    print(f'\n{"─"*60}')
    print(f'  Country    : {COUNTRY}')
    print(f'  Variable   : {_var}')
    print(f'  Timesteps  : {len(era5_long_term.time)}  '
          f'({pd.Timestamp(era5_long_term.time.values[0]).year} – '
          f'{pd.Timestamp(era5_long_term.time.values[-1]).year})')
    print(f'  Temp range : {float(era5_long_term[_var].min()):.1f}°C – '
          f'{float(era5_long_term[_var].max()):.1f}°C')
    print(f'{"─"*60}\n')

    def plot_monthly_heatmap(ds, variable, country, cmap='coolwarm', save_dir=None):
        spatial_dims = [d for d in ds[variable].dims if d not in ['time', 'date']]
        monthly_mean = ds[variable].mean(dim=spatial_dims).to_series()
        monthly_mean.index = pd.MultiIndex.from_arrays(
            [monthly_mean.index.year, monthly_mean.index.month],
            names=['year', 'month']
        )
        heatmap_data = monthly_mean.unstack(level='year')

        fig, ax = plt.subplots(figsize=(max(12, len(heatmap_data.columns) * 0.4), 6))
        sns.heatmap(
            heatmap_data, cmap=cmap, ax=ax,
            cbar_kws={'label': f'Mean {variable} (°C)', 'shrink': 0.8},
            linewidths=0.3, linecolor='white'
        )
        ax.set_title(f'{country} — ERA5 Monthly Mean Temperature '
                     f'({heatmap_data.columns[0]}–{heatmap_data.columns[-1]})',
                     fontsize=13)
        ax.set_xlabel('Year', fontsize=10)
        ax.set_ylabel('Month', fontsize=10)
        ax.set_yticklabels(
            ['Jan','Feb','Mar','Apr','May','Jun',
             'Jul','Aug','Sep','Oct','Nov','Dec'],
            rotation=0, fontsize=9
        )
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
        plt.tight_layout()
        if save_dir:
            out = os.path.join(save_dir, f'era5_monthly_heatmap_{country}.png')
            plt.savefig(out, dpi=150, bbox_inches='tight')
            print(f'  💾 Saved: {out}')
        plt.show()

    def plot_annual_trend(ds, variable, country, save_dir=None):
        spatial_dims = [d for d in ds[variable].dims if d not in ['time', 'date']]
        annual = (ds[variable]
                  .mean(dim=spatial_dims)
                  .to_series()
                  .resample('YE').mean())

        x = np.arange(len(annual))
        m, b = np.polyfit(x, annual.values, 1)

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(annual.index, annual.values,
                color='steelblue', linewidth=1.5, label='Annual mean')
        ax.plot(annual.index, m * x + b,
                color='red', linestyle='--', linewidth=1.2,
                label=f'Trend: {m*10:.3f} °C/decade')
        ax.fill_between(annual.index, annual.values,
                        alpha=0.15, color='steelblue')
        ax.set_title(f'{country} — ERA5 annual mean temperature trend '
                     f'({annual.index[0].year}–{annual.index[-1].year})',
                     fontsize=12)
        ax.set_ylabel('°C')
        ax.set_xlabel('Year')
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        if save_dir:
            out = os.path.join(save_dir, f'era5_annual_trend_{country}.png')
            plt.savefig(out, dpi=150, bbox_inches='tight')
            print(f'   Saved: {out}')
        plt.show()
        print(f'  Trend: {m*10:.3f} °C per decade  '
              f'({m*42:.2f} °C total over 42 years)')

    RESULTS_DIR = f'./results/SkillExplorer_{COUNTRY}_{pd.Timestamp.today().date()}'
    os.makedirs(RESULTS_DIR, exist_ok=True)

    plot_monthly_heatmap(era5_long_term, _var, COUNTRY, cmap='coolwarm', save_dir=RESULTS_DIR)
    plot_annual_trend(era5_long_term, _var, COUNTRY, save_dir=RESULTS_DIR)

    print(f'\n✅ Step 7 complete — {COUNTRY}')